In [ ]:
!pip install --upgrade gradio

^C


  Using cached gradio-6.9.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.135.1-py3-none-any.whl.metadata (30 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.3.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached orjson-3.11.7-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached pillow-12.1.1-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.22-py3-none-any.whl.metadata (1.8 kB)
  Using cached safehttpx-0.1.7-py3-none-any

ERROR: Could not install packages due to an OSError: [WinError 32] プロセスはファイルにアクセスできません。別のプロセスが使用中です。: 'C:\\Users\\kobar\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\pydantic\\_internal\\_validate_call.py'
Consider using the `--user` option or check the permissions.



In [4]:
import gradio as gr
from PIL import Image, ImageDraw
import copy
import pandas as pd
import ast

#初期値の設定
START_POS = [100, 175]


#名前、説明文、座標を辞書で管理する
LOCATION_DATA = {}

df = pd.read_csv('location_data.csv') # CSVファイルをDataFrameとして読み込む
df['coor'] = df['coor'].apply(ast.literal_eval)  #文字列をリスト型に変換
df.set_index('id', inplace=True)
LOCATION_DATA = df.to_dict(orient='index') # DataFrameを辞書形式に変換


#画像の読み込み
with Image.open("school_map_image.jpg") as img:
  IMG_SIZE = img.size


#今いる現在地を探す関数
def get_location(pos):
  x,y = pos

  #辞書を全て参照し、該当するエリアを判定し探す
  for location in LOCATION_DATA.values():
      coor = location["coor"]
      if coor[0] < x < coor[1] and coor[2] < y < coor[3]:
          return location["name"], location["description"]

  #どのエリアにも当てはまらない場合
  return "通路", "校舎や施設をつなぐ道です。次の目的地へ向かいましょう。"


#軌跡と現在地の点を背景画像の上に描画する関数
def draw_map(path):
  img = Image.open("school_map_image.jpg").copy()
  draw = ImageDraw.Draw(img)

  if len(path) > 1: #pathのリストに要素が二つ入っていればそれらを線で結ぶ
      draw.line(path,fill="red",width=3)

  x,y = path[-1] #現在地の座標を格納する
  box = [x-8,y-8,x+8,y+8]
  draw.ellipse(box,fill="yellow",outline="black") #boxの範囲内に円を作成する

  return img


#通路や建物などの特定の場所で止まるようにする関数
def stop_location(x,y,x2,y2):
  if x>=255 and x2<255 or x<=255 and x2>255: x=255 #メンスト
  if x>=305 and x2<305 or x<=305 and x2>305: x=305 #体育館
  if x>=435 and x2<435 or x<=435 and x2>435: x=435 #図書館
  if x>=465 and x2<465 or x<=465 and x2>465: x=465 #10号館
  if x>=495 and x2<495 or x<=495 and x2>495: x=495 #図書館角
  if y>=145 and y2<145 or y<=145 and y2>145: y=145 #図書館・11号館
  if y>=175 and y2<175 or y<=175 and y2>175: y=175 #メンスト
  if y>=200 and y2<200 or y<=200 and y2>200: y=200 #10号館
  if y>=230 and y2<230 or y<=230 and y2>230: y=230 #体育館

  return x,y


#押されたボタンによって現在地の座標を更新し、各関数を実行する
def update_coor(coor_history, direction, STEP):
  current_pos = coor_history[-1]
  x,y = current_pos
  x2=copy.copy(x)
  y2=copy.copy(y)

  if direction == "リセット": #リセットボタンを押された場合
    new_coor = [START_POS]
  else: #それぞれの動き方に合わせて座標を動かす
    if direction == "▲ 上": y -= STEP
    elif direction == "▼ 下": y += STEP
    elif direction == "◀ 左": x -= STEP
    elif direction == "▶ 右": x += STEP

    x,y=stop_location(x,y,x2,y2) #止まる座標を関数で呼び出す

    x = max(0,min(x,IMG_SIZE[0])) #0以上、画像の幅以下に指定する
    y = max(0,min(y,IMG_SIZE[1]))
    new_pos = [x,y]
    new_coor = coor_history + [new_pos] #座標の更新

  new_image = draw_map(new_coor) #軌跡を引く関数を呼び出す
  location_name,location_desc = get_location(new_coor[-1]) #

  return new_coor,new_image,location_name,location_desc


#GUIの作成
with gr.Blocks() as demo:
  gr.Markdown("# SophiNavi📍")
  gr.Markdown("方向ボタンで点を動かすと、軌跡と現在地が更新されます。")

  coor_state = gr.State(value=[START_POS]) #Stateの更新
  name,desc = get_location(START_POS) #現在地の場所を関数で呼び出す

  with gr.Row():
    map_image=gr.Image(value=draw_map([START_POS]),label="上智大学のマップ",interactive=False)

    with gr.Column():
      step_slider = gr.Slider(minimum=10,maximum=100,value=55,step=5,label="移動歩数",info="一度に動く距離を設定してください") #歩数スライダーの設定

      gr.Markdown("動きたい方向のキーを押してください") #十字キーの設定
      with gr.Row():
        gr.Markdown("")
        btn_up = gr.Button("▲ 上")
        gr.Markdown("")
      with gr.Row():
        btn_left = gr.Button("◀ 左")
        btn_right = gr.Button("▶ 右")
      with gr.Row():
        gr.Markdown("")
        btn_down = gr.Button("▼ 下")
        gr.Markdown("")
      with gr.Row():
        btn_reset = gr.Button("リセット", variant="stop")

    name_display = gr.Textbox(value=name, label="現在地", interactive=False)
    desc_display = gr.Textbox(value=desc, label="場所の説明", lines=5, interactive=False)

    #outputsのlist設定
    outputs_list = [coor_state,map_image,name_display,desc_display]

    #クリックしたときの設定
    btn_up.click(update_coor, inputs=[coor_state, gr.Textbox(value="▲ 上", visible=False),step_slider], outputs=outputs_list)
    btn_down.click(update_coor, inputs=[coor_state, gr.Textbox(value="▼ 下", visible=False),step_slider], outputs=outputs_list)
    btn_left.click(update_coor, inputs=[coor_state, gr.Textbox(value="◀ 左", visible=False),step_slider], outputs=outputs_list)
    btn_right.click(update_coor, inputs=[coor_state, gr.Textbox(value="▶ 右", visible=False),step_slider], outputs=outputs_list)
    btn_reset.click(update_coor, inputs=[coor_state, gr.Textbox(value="リセット", visible=False),step_slider], outputs=outputs_list)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fe9af2a367a067cb61.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
